# Study 932 — Trust Yield — the teardown

The position-level excess over cash, the cluster bootstrap over shells, the one-bet-per-name cut, the two daily books, the era cut, four assumption sweeps, two cross-checks, the yield path, and the live synthetic control that shows why the naive *t* must not be quoted. Real numbers are frozen from `docs/results.md` (fingerprint `7d2c2f7b9919`, cash `b9dc0d8fa0bf`, as-of 2026-06-30).

In [1]:
R = {'asof': '2026-06-30', 'fp': '7d2c2f7b9919', 'fp_cash': 'b9dc0d8fa0bf', 'tape_start': '2019-03-27', 'tape_end': '2024-04-10', 'n_listed': 31, 'n_window': 31, 'n_traded': 25, 'n_pos': 203, 'hold_days': 302, 'bad_prints': 20, 'bad_name': 'CXAI', 'disc': 1.47, 'ytr': 4.03, 'exc': 1.34, 'exc_ann': 2.31, 'hit': 93.1, 't': 17.87, 't_hac': 17.96, 'ci_lo': 1.04, 'ci_hi': 1.63, 'ci_neg': 0.0, 'geo_ann': 1.62, 'book_cagr': 1.23, 'book_gross': 1.39, 'one_n': 25, 'one_exc': 1.45, 'one_ann': 1.61, 'one_t': 7.46, 'one_lo': 1.09, 'one_hi': 1.83, 'one_hit': 100, 'best_exc': 10.99, 'best_lo': 4.66, 'best_hi': 18.02, 'best_t': 9.12, 'ident_resid': '3.3e-04', 'ident_corr': 0.9992, 'anchor_med': 0.68, 'anchor_above': 14, 'anchor_within2': 14, 'anchor_worst': -2.87, 'anchor_worst_name': 'ALTI', 'anchor_below_n': 11, 'anchor_below_mean': -0.81, 'sell_exc': 10.43, 'sell_t': 8.49, 'sell_lo': 3.96, 'sell_hi': 17.7, 'sell_one_exc': 10.99, 'sell_one_t': 2.54, 'sell_one_lo': 4.05, 'sell_one_hi': 19.88, 'sell_one_hit': 80, 'floor_exc': 0.83, 'floor_t': 9.29, 'floor_lo': 0.28, 'floor_hi': 1.28, 'floor_hit': 78, 'floor_book': 0.86, 'floor_one_exc': 1.09, 'floor_one_t': 5.4, 'floor_one_lo': 0.7, 'floor_one_hi': 1.47, 'la_n': 218, 'la_exc': 1.36, 'la_lo': 1.06, 'la_hi': 1.63, 'acc_sharpe': 5.49, 'acc_cagr': 1.23, 'acc_vol': 0.22, 'acc_dd': -0.24, 'acc_t': 13.25, 'mtm_sharpe': 0.36, 'mtm_cagr': 6.91, 'mtm_vol': 37.34, 'mtm_dd': -41.2, 'mtm_t': 0.95, 'mtm_ci_lo': -0.31, 'mtm_ci_hi': 0.92, 'mtm_neg': 13.8, 'era_a_n': 124, 'era_a_s': 24, 'era_a_disc': 1.4, 'era_a_exc': 1.25, 'era_a_ann': 1.98, 'era_a_t': 14.52, 'era_a_lo': 0.98, 'era_a_hi': 1.52, 'era_a_hit': 94, 'era_b_n': 79, 'era_b_s': 9, 'era_b_disc': 1.57, 'era_b_exc': 1.47, 'era_b_ann': 2.83, 'era_b_t': 10.8, 'era_b_lo': 0.79, 'era_b_hi': 2.01, 'era_b_hit': 92, 'cost': ((0, 1.49, 1.39, 19.88, 95), (15, 1.34, 1.23, 17.87, 93), (50, 0.98, 0.84, 13.16, 82), (100, 0.47, 0.3, 6.4, 65), (200, -0.52, -0.78, -7.19, 31)), 'trust': ((9.9, 130, 0.83, 0.56, 10.51), (10.0, 203, 1.34, 1.23, 17.87), (10.1, 241, 2.05, 2.11, 25.58), (10.2, 251, 2.97, 3.19, 34.95)), 'buffer': ((15, 1.31, 1.22), (30, 1.34, 1.23), (60, 1.41, 1.33), (90, 1.42, 1.38)), 'fee': ((0, 1.34, 1.23), (10, 1.23, 1.08), (25, 1.03, 0.83)), 'guard_note': 'identical at +1.34% on 6% / 12% / 25% / off', 'sgov_exc': 1.42, 'sgov_ann': 2.75, 'sgov_t': 16.91, 'sgov_n': 179, 'drop_exc': 1.4, 'drop_ann': 2.4, 'drop_t': 18.28, 'drop_n': 189, 'peak_month': '2022-08', 'peak_disc': 2.03, 'peak_ytr': 7.06, 'peak_bill': 2.86, 'mania_month': '2021-02', 'mania_disc': -9.44, 'mania_live': 13, 'live_2021': 16, 'live_2022': 5, 'live_2024': 1, 'syn_seeds': 16, 'syn_pl_mean': 1.938, 'syn_pl_sd': 0.144, 'syn_pl_t': 16, 'syn_pl_ci': 16, 'syn_nl_mean': 0.002, 'syn_nl_sd': 1.585, 'syn_nl_t': 6, 'syn_nl_ci': 1}

## Construction

- **Tape.** Pre-deal SPAC quotes are **unadjusted** closes read off each SPAC's *successor* ticker (Yahoo carries the pre-deal history forward). Unadjusted is deliberate: a $10 trust is a dollar quantity, and a post-deal reverse split would rescale the pre-deal tape into nonsense. Names that split post-deal are therefore **absent** — a documented selection.
- **Trust path (PROXY).** $10.00 at IPO accreting at BIL's total return, less an optional fee drag. Swept.
- **Deadline (ASSUMPTION).** Hardcoded deal close **− 30 days**, because the redemption election is due before the vote and the vote before the close. Swept. Without the buffer several shells de-anchor violently in the final fortnight — which is the trust put expiring, exactly on cue.
- **Execution.** Signal on the month-end close, buy at the **next** close. One lag. 15 bps one-way × NAV at entry; redemption is at trust and free. No shorts, so no borrow leg.
- **Cleaning.** 20 pre-deal quotes below 60% of trust (all in CXAI, a 2023 vendor artefact) are replaced by the previous good quote — inside the pre-deal window only. Dropping that name entirely *raises* the headline.

## The headline

In [2]:
print(f"{R['n_pos']} positions across {R['n_traded']} SPACs (of {R['n_listed']} listed); mean hold {R['hold_days']:.0f} days")
print(f"mean discount at entry      {R['disc']:+.2f}%")
print(f"mean implied YTR at entry   {R['ytr']:+.2f}%")
print(f"mean excess over cash       {R['exc']:+.2f}% per position")
print(f"hit rate {R['hit']:.1f}%   one-sample t {R['t']:+.2f}   HAC t {R['t_hac']:+.2f}")
print(f"cluster bootstrap over {R['n_traded']} shells: 95% CI [{R['ci_lo']:+.2f}%, {R['ci_hi']:+.2f}%], share<0 {R['ci_neg']:.1f}%")
print()
print(f"one position per SPAC (n={R['one_n']}): {R['one_exc']:+.2f}%, t {R['one_t']:+.2f}, CI [{R['one_lo']:+.2f}%, {R['one_hi']:+.2f}%], hit {R['one_hit']}%")
print(f"redeem-or-sell upper bound : {R['best_exc']:+.2f}% CI [{R['best_lo']:+.2f}%, {R['best_hi']:+.2f}%] t {R['best_t']:+.2f}")

203 positions across 25 SPACs (of 31 listed); mean hold 302 days
mean discount at entry      +1.47%
mean implied YTR at entry   +4.03%
mean excess over cash       +1.34% per position
hit rate 93.1%   one-sample t +17.87   HAC t +17.96
cluster bootstrap over 25 shells: 95% CI [+1.04%, +1.63%], share<0 0.0%

one position per SPAC (n=25): +1.45%, t +7.46, CI [+1.09%, +1.83%], hit 100%
redeem-or-sell upper bound : +10.99% CI [+4.66%, +18.02%] t +9.12


> 💡 **In plain words.** Twenty-five shells, every one of them a winner, for about a percent and a third apiece. The 'redeem-or-sell' line is what you would have made if you had been allowed to look at the deadline-day quote and take the better of it and the trust — eight times more. We do **not** claim that number: the redemption election is filed days in advance, so the rule redeems unconditionally and gives up every deal premium on offer. The headline is the conservative branch.

## Now read that table again: it is an identity

The payoff above is **imposed** — the model hands a redeemed share its accrued trust — and the benchmark is the same cash leg the trust is assumed to accrue at. Put those two together and the position excess collapses to

```
excess  =  (1 + cash_ret) x (trust at signal / cost-loaded entry price - 1)
```

i.e. **to the entry discount**. The cell below checks that on the real positions.

In [3]:
print(f"max |excess - (1+cash) x entry discount| over all {R['n_pos']} "
      f"positions : {R['ident_resid']}")
print(f"correlation of the excess with the entry discount    : {R['ident_corr']:.4f}")
print()
print('=> the +%.2f%%, the %.1f%% hit rate, the t of %+.1f and the cluster CI are'
      % (R['exc'], R['hit'], R['t']))
print('   all statistics about THE ENTRY DISCOUNT - about whether a discount')
print('   selected on Monday was still there on Tuesday. None of them is')
print('   evidence that the redemption paid: that is an assumption, not a')
print('   measurement. The one-per-name t of %+.2f is the same identity with'
      % R['one_t'])
print('   fewer observations.')

max |excess - (1+cash) x entry discount| over all 203 positions : 3.3e-04
correlation of the excess with the entry discount    : 0.9992

=> the +1.34%, the 93.1% hit rate, the t of +17.9 and the cluster CI are
   all statistics about THE ENTRY DISCOUNT - about whether a discount
   selected on Monday was still there on Tuesday. None of them is
   evidence that the redemption paid: that is an assumption, not a
   measurement. The one-per-name t of +7.46 is the same identity with
   fewer observations.


That is not a scandal — a redemption right *is* a contractual identity, and this desk has stamped mechanical identities Real before. But it means the Signal stamp has to be earned somewhere the model is not doing the talking. Three places:

**(a) Did the market agree with our trust line?** On the day the redemption right expired, where was the shell actually quoted, against the $10.00-accreted-at-BIL line we assumed?

In [4]:
print(f"median gap (quote - assumed trust)      : {R['anchor_med']:+.2f}%")
print(f"shells quoting at or above the line     : {R['anchor_above']}/{R['n_traded']}")
print(f"shells within +/-2% of it               : {R['anchor_within2']}/{R['n_traded']}")
print(f"worst shell                             : {R['anchor_worst']:+.2f}% ({R['anchor_worst_name']})")
print(f"mean shortfall on the {R['anchor_below_n']} shells below : {R['anchor_below_mean']:+.2f}%")

median gap (quote - assumed trust)      : +0.68%
shells quoting at or above the line     : 14/25
shells within +/-2% of it               : 14/25
worst shell                             : -2.87% (ALTI)
mean shortfall on the 11 shells below : -0.81%


No shell's quote fell more than **2.9%** under the assumed line. The trust value we guessed is roughly where the market itself was when the put expired.

**(b) The assumption-free version.** *Sell* into that deadline quote instead of redeeming, so the trust never enters the payoff at all:

In [5]:
print(f"all positions : {R['sell_exc']:+.2f}%  t {R['sell_t']:+.2f}  CI [{R['sell_lo']:+.2f}%, {R['sell_hi']:+.2f}%]")
print(f"one per shell : {R['sell_one_exc']:+.2f}%  t {R['sell_one_t']:+.2f}  CI [{R['sell_one_lo']:+.2f}%, {R['sell_one_hi']:+.2f}%]  hit {R['sell_one_hit']}%")
print()
print('clears |t| >= 2 on the near-independent cut with NO trust assumption -')
print('but it is a different, far wilder trade (its return is de-SPAC hype:')
print('one shell quoted +100% over trust on its deadline). Corroboration,')
print('not the headline.')

all positions : +10.43%  t +8.49  CI [+3.96%, +17.70%]
one per shell : +10.99%  t +2.54  CI [+4.05%, +19.88%]  hit 80%

clears |t| >= 2 on the near-independent cut with NO trust assumption -
but it is a different, far wilder trade (its return is de-SPAC hype:
one shell quoted +100% over trust on its deadline). Corroboration,
not the headline.


**(c) The adversarial payoff.** Let the tape veto our assumption shell by shell: pay the **worse** of the assumed accrued trust and the deadline-day quote (`market_floor=True`). Deliberately too harsh — the redemption right pays trust whatever the screen says — which is the point of a floor:

In [6]:
print(f"all positions : {R['floor_exc']:+.2f}%  t {R['floor_t']:+.2f}  CI [{R['floor_lo']:+.2f}%, {R['floor_hi']:+.2f}%]  hit {R['floor_hit']}%  book {R['floor_book']:+.2f}%/yr")
print(f"one per shell : {R['floor_one_exc']:+.2f}%  t {R['floor_one_t']:+.2f}  CI [{R['floor_one_lo']:+.2f}%, {R['floor_one_hi']:+.2f}%]")
print()
print('the edge survives the harshest haircut available on its own')
print('load-bearing assumption, at roughly two thirds of its size.')

all positions : +0.83%  t +9.29  CI [+0.28%, +1.28%]  hit 78%  book +0.86%/yr
one per shell : +1.09%  t +5.40  CI [+0.70%, +1.47%]

the edge survives the harshest haircut available on its own
load-bearing assumption, at roughly two thirds of its size.


## Annualising it honestly

Three numbers, only one of which is a rate of return:

In [7]:
print(f"mean of the per-position annualised excess : {R['exc_ann']:+.2f}%  "
      f"<- over-weights short holds; NOT a rate of return")
print(f"the mean position over its {R['hold_days']:.0f}-day hold      : {R['geo_ann']:+.2f}%")
print(f"THE BOOK (equal-weighted, net of cost)     : {R['book_cagr']:+.2f}%")
print()
print('Drop the 30-day minimum horizon and the first number goes to +112%,')
print('which is all you need to know about quoting it. The book is the')
print('bankable figure and it is what the Tradability card uses.')

mean of the per-position annualised excess : +2.31%  <- over-weights short holds; NOT a rate of return
the mean position over its 302-day hold      : +1.62%
THE BOOK (equal-weighted, net of cost)     : +1.23%

Drop the 30-day minimum horizon and the first number goes to +112%,
which is all you need to know about quoting it. The book is the
bankable figure and it is what the Tradability card uses.


## Look-ahead, named

The deadline and the 30-730-day horizon filter are built from the **realised** deal date, which nobody knew at entry. (A real holder would have redeemed against the *charter* deadline, which was known ex ante.) Lifting the horizon cap to 3000 days moves nothing:

In [8]:
print(f"max_days= 730 : n={R['n_pos']}  excess {R['exc']:+.2f}%  CI [{R['ci_lo']:+.2f}%, {R['ci_hi']:+.2f}%]")
print(f"max_days=3000 : n={R['la_n']}  excess {R['la_exc']:+.2f}%  CI [{R['la_lo']:+.2f}%, {R['la_hi']:+.2f}%]")
print('\nthe look-ahead is present and inert.')

max_days= 730 : n=203  excess +1.34%  CI [+1.04%, +1.63%]
max_days=3000 : n=218  excess +1.36%  CI [+1.06%, +1.63%]

the look-ahead is present and inert.


## Why the naive *t* is not the number

Monthly entries in the same shell are the same bet on the same deadline: the same trust, the same date, largely the same discount. The independent unit is the **name**. The cluster bootstrap resamples whole shells; the synthetic null below quantifies what the naive interval costs.

## The two daily books (excess-of-cash)

In [9]:
print(f"accrual (hold-to-redemption): exSharpe {R['acc_sharpe']:+.2f}  exCAGR {R['acc_cagr']:+.2f}%  vol {R['acc_vol']:.2f}%  maxDD {R['acc_dd']:+.2f}%  HAC t {R['acc_t']:+.2f}")
print(f"mark-to-market             : exSharpe {R['mtm_sharpe']:+.2f}  exCAGR {R['mtm_cagr']:+.2f}%  vol {R['mtm_vol']:.2f}%  maxDD {R['mtm_dd']:+.2f}%  HAC t {R['mtm_t']:+.2f}")
print(f"mark-to-market Sharpe 95% CI [{R['mtm_ci_lo']:+.2f}, {R['mtm_ci_hi']:+.2f}]  share<0 {R['mtm_neg']:.1f}%")

accrual (hold-to-redemption): exSharpe +5.49  exCAGR +1.23%  vol 0.22%  maxDD -0.24%  HAC t +13.25
mark-to-market             : exSharpe +0.36  exCAGR +6.91%  vol 37.34%  maxDD -41.20%  HAC t +0.95
mark-to-market Sharpe 95% CI [-0.31, +0.92]  share<0 13.8%


The accrual Sharpe of **+5.49** is an artefact of the construction, not a claim: once a position is on, the pull to trust is deterministic, so the only variance left is the changing mix of open positions. The mark-to-market book is the honest path, and its Sharpe CI **[-0.31, +0.92]** straddles zero. Most of its 37% vol and 41% drawdown is the *modelled forced redemption*: shells quoting a deal premium on the deadline day get marked down to trust in a single print. That is forgone upside recognised at once, not a loss — and it is the cost of the conservative branch we chose.

## Era cut (entry year, split 2022)

In [10]:
print(f"2019-2021 (zero rates) : n={R['era_a_n']:3d} ({R['era_a_s']} shells)  disc {R['era_a_disc']:+.2f}%  excess {R['era_a_exc']:+.2f}% ({R['era_a_ann']:+.2f}% ann)  t {R['era_a_t']:+.2f}  CI [{R['era_a_lo']:+.2f}%, {R['era_a_hi']:+.2f}%]  hit {R['era_a_hit']}%")
print(f"2022-2024 (hiking)     : n={R['era_b_n']:3d} ({R['era_b_s']} shells)  disc {R['era_b_disc']:+.2f}%  excess {R['era_b_exc']:+.2f}% ({R['era_b_ann']:+.2f}% ann)  t {R['era_b_t']:+.2f}  CI [{R['era_b_lo']:+.2f}%, {R['era_b_hi']:+.2f}%]  hit {R['era_b_hit']}%")

2019-2021 (zero rates) : n=124 (24 shells)  disc +1.40%  excess +1.25% (+1.98% ann)  t +14.52  CI [+0.98%, +1.52%]  hit 94%
2022-2024 (hiking)     : n= 79 (9 shells)  disc +1.57%  excess +1.47% (+2.83% ann)  t +10.80  CI [+0.79%, +2.01%]  hit 92%


Positive in both, wider and faster in the hiking era — but the late era rests on **9** shells. The opportunity set was already collapsing while the yield was at its best.

## Sweep 1 — cost (one-way on NAV at entry; redemption free)

In [11]:
print(f"   0 bps : excess +1.49%/position  book +1.39%/yr  t +19.88  hit 95%")
print(f"  15 bps : excess +1.34%/position  book +1.23%/yr  t +17.87  hit 93%")
print(f"  50 bps : excess +0.98%/position  book +0.84%/yr  t +13.16  hit 82%")
print(f" 100 bps : excess +0.47%/position  book +0.30%/yr  t +6.40  hit 65%")
print(f" 200 bps : excess -0.52%/position  book -0.78%/yr  t -7.19  hit 31%")

   0 bps : excess +1.49%/position  book +1.39%/yr  t +19.88  hit 95%
  15 bps : excess +1.34%/position  book +1.23%/yr  t +17.87  hit 93%
  50 bps : excess +0.98%/position  book +0.84%/yr  t +13.16  hit 82%
 100 bps : excess +0.47%/position  book +0.30%/yr  t +6.40  hit 65%
 200 bps : excess -0.52%/position  book -0.78%/yr  t -7.19  hit 31%


## Sweep 2 — the trust level, the load-bearing assumption

In [12]:
print(f"$9.90 : n=130  excess +0.83%/position  book +0.56%/yr  t +10.51")
print(f"$10.00 : n=203  excess +1.34%/position  book +1.23%/yr  t +17.87")
print(f"$10.10 : n=241  excess +2.05%/position  book +2.11%/yr  t +25.58")
print(f"$10.20 : n=251  excess +2.97%/position  book +3.19%/yr  t +34.95")

$9.90 : n=130  excess +0.83%/position  book +0.56%/yr  t +10.51
$10.00 : n=203  excess +1.34%/position  book +1.23%/yr  t +17.87
$10.10 : n=241  excess +2.05%/position  book +2.11%/yr  t +25.58
$10.20 : n=251  excess +2.97%/position  book +3.19%/yr  t +34.95


Everything scales with a number we assumed rather than filed: a dime of trust moves the result by more than the result itself. $10.00 is the standard 2020-2021 trust funding, but over-funded shells at $10.10-$10.20 existed, and extension votes sometimes topped the trust up. **$9.90 is the honest floor** and it still clears zero (+0.56%/yr as a book) — and the market's own shell-by-shell answer, the `market_floor` cut above, lands at +0.86%/yr. That is what keeps the Signal stamp green.

## Sweeps 3 and 4 — redemption buffer and trust fee drag

In [13]:
print(f" 15d : excess +1.31%/position  book +1.22%/yr")
print(f" 30d : excess +1.34%/position  book +1.23%/yr")
print(f" 60d : excess +1.41%/position  book +1.33%/yr")
print(f" 90d : excess +1.42%/position  book +1.38%/yr")
print()
print(f"  0bp : excess +1.34%/position  book +1.23%/yr")
print(f" 10bp : excess +1.23%/position  book +1.08%/yr")
print(f" 25bp : excess +1.03%/position  book +0.83%/yr")
print()
print('deep-quote guard: identical at +1.34% on 6% / 12% / 25% / off')

 15d : excess +1.31%/position  book +1.22%/yr
 30d : excess +1.34%/position  book +1.23%/yr
 60d : excess +1.41%/position  book +1.33%/yr
 90d : excess +1.42%/position  book +1.38%/yr

  0bp : excess +1.34%/position  book +1.23%/yr
 10bp : excess +1.23%/position  book +1.08%/yr
 25bp : excess +1.03%/position  book +0.83%/yr

deep-quote guard: identical at +1.34% on 6% / 12% / 25% / off


The guard never binds: once the vendor artefacts are cleaned, **no** entry in the whole sample implied a discount deeper than 6%. This was always a small, patient trade, never a distressed one.

## Cross-checks

In [14]:
print(f"SGOV instead of BIL as cash/accrual leg : {R['sgov_exc']:+.2f}% ({R['sgov_ann']:+.2f}% ann)  t {R['sgov_t']:+.2f}  n={R['sgov_n']}")
print(f"drop the print-cleaned name ({R['bad_name']})      : {R['drop_exc']:+.2f}% ({R['drop_ann']:+.2f}% ann)  t {R['drop_t']:+.2f}  n={R['drop_n']}")

SGOV instead of BIL as cash/accrual leg : +1.42% (+2.75% ann)  t +16.91  n=179
drop the print-cleaned name (CXAI)      : +1.40% (+2.40% ann)  t +18.28  n=189


## The yield path — when the trade existed

In [15]:
print(f"{R['mania_month']} (the mania): median shell {abs(R['mania_disc']):.1f}% "
      f"ABOVE trust across {R['mania_live']} live shells — no trade")
print(f"{R['peak_month']} (the peak) : median discount {R['peak_disc']:+.2f}%, "
      f"implied YTR {R['peak_ytr']:+.2f}% vs a {R['peak_bill']:.2f}% 3m bill")
print(f"live pre-deal shells: {R['live_2021']} (early 2021) -> "
      f"{R['live_2022']} (end 2022) -> {R['live_2024']} (2024)")

2021-02 (the mania): median shell 9.4% ABOVE trust across 13 live shells — no trade
2022-08 (the peak) : median discount +2.03%, implied YTR +7.06% vs a 2.86% 3m bill
live pre-deal shells: 16 (early 2021) -> 5 (end 2022) -> 1 (2024)


## Live synthetic control — the plant, the null, and the cost of the naive *t*

**Offline synthetic, not the tape.** The null cannot be a smaller discount — buying below a line and being paid that line is arithmetic. So the null breaks the *mechanism*: quotes are an unanchored martingale accruing at the cash rate, and the terminal payoff is whatever the last quote happens to be. Run live over 16 seeds, scoring both the naive position-level *t* and the cluster bootstrap.

In [16]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import warnings; warnings.filterwarnings('ignore')
import numpy as np
from trust_yield import data, strategy as st
for ss, label in [(1.0, 'put binds      '), (0.0, 'put is fiction ')]:
    outs = [st.synthetic_detect(*data.synthetic_panel(signal_strength=ss, seed=s,
                                                     n_days=700)[:3],
                                n_boot=800)
            for s in range(932, 948)]
    m = np.array([o['mean_excess'] for o in outs])
    t = np.array([o['t_pos'] for o in outs])
    ci = sum(1 for o in outs if o['ci_low'] > 0)
    print(f"{label}: mean excess {m.mean():+.3%} (sd {m.std(ddof=1):.3%})  "
          f"naive t>2 on {int((t>2).sum()):2d}/16   cluster CI>0 on {ci:2d}/16")

put binds      : mean excess +1.907% (sd 0.147%)  naive t>2 on 16/16   cluster CI>0 on 16/16


put is fiction : mean excess +0.282% (sd 1.472%)  naive t>2 on  6/16   cluster CI>0 on  1/16


The frozen 24-shell version in `docs/results.md`: the plant is recovered **16/16** times (mean +1.938%), the null is dead-centred (mean +0.002%, sd 1.585%) — and the naive position-level *t* false-fires on **6/16** null panels where the cluster bootstrap fires on **1/16**, right at its nominal 5%. That is the whole argument for the interval this study quotes.

## Live recomputation from the shared cache (labelled)

**Real tape.** This cell recomputes the headline from `studies/_cache` and checks it against the frozen dict. If the cache is absent it says so and computes nothing — it never substitutes synthetic numbers under this banner.

In [17]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import warnings; warnings.filterwarnings('ignore')
from trust_yield import data, strategy as st
if not data.have_real():
    print('shared _cache absent — real-tape recomputation skipped '
          '(run examples/verify.py --fetch to populate it). No numbers shown.')
else:
    px, bad = data.clean_quotes(data.load_spac_closes())
    cash = data.load_cash()['BIL'].dropna()
    live = st.race(px, cash, cost_bps=15.0)
    live_book = st.summary(st.portfolio_daily(live['positions'], px, cash,
                                              mode='accrual'))
    print(f"live : {live['n_pos']} positions / {live['n_spacs']} shells   "
          f"excess {live['mean_excess']:+.2%}   book {live_book['cagr']:+.2%}/yr   "
          f"CI [{live['boot']['ci_low']:+.2%}, {live['boot']['ci_high']:+.2%}]")
    print(f"frozen: {R['n_pos']} positions / {R['n_traded']} shells   "
          f"excess {R['exc']/100:+.2%}   book {R['book_cagr']/100:+.2%}/yr   "
          f"CI [{R['ci_lo']/100:+.2%}, {R['ci_hi']/100:+.2%}]")
    print(f"identity residual: {live['identity']['max_abs_residual']:.1e}  "
          f"corr with entry discount {live['identity']['corr_with_discount']:.4f}")
    print('bad prints cleaned:', bad)
    print('fingerprint:', data.fingerprint(px), '(frozen', R['fp'] + ')')

live : 203 positions / 25 shells   excess +1.34%   book +1.23%/yr   CI [+1.04%, +1.63%]
frozen: 203 positions / 25 shells   excess +1.34%   book +1.23%/yr   CI [+1.04%, +1.63%]
identity residual: 3.3e-04  corr with entry discount 0.9992
bad prints cleaned: {'CXAI': 20}
fingerprint: 7d2c2f7b9919 (frozen 7d2c2f7b9919)


## Verdict

- **Signal — Real.** Mean excess over cash of **+1.34%** per 302-day position, cluster CI **[+1.04%, +1.63%]** over 25 shells with zero negative resamples, positive in both eras, robust to the buffer, the fee drag and the quote guard. **That headline is an identity** — the payoff is imposed, so its *t* of +17.9 (and the one-per-name +7.46) describe the entry discount, not the redemption. The stamp rests on the three unassumed reads: the deadline quote at or above the assumed trust in 14/25 shells and never more than 2.9% below; the sell-at-quote version clearing *t* = +2.54 one-per-name with no trust assumption at all; and the `market_floor` payoff still paying +0.83% (CI [+0.28%, +1.28%]). The mechanism is visible in the path too: implied YTR of 7.1% against a 2.9% bill at the peak. **Survivorship named:** successor-ticker, no-reverse-split shells only, so liquidations and split names are absent and the opportunity count is flattered; the payoff itself is unaffected, since a liquidation pays the trust too.
- **Tradability — Fragile.** +1.2% a year over bills **as a book** (the +2.31% mean-of-annualised is not a rate of return) for a 302-day lock-up in a $200-300m shell trading a few hundred thousand dollars a day; dead at 200 bps of friction (-0.78%/yr); more than halved if the trust was $9.90 (+0.56%/yr); the broker's fee for filing the redemption election is not modelled at all; dependent on a deadline sponsors kept moving; and gone — 16 live shells in 2021, 1 by 2024. A sound mechanism you could not size, in a market that closed behind it.